Week 13 · Day 1 — GPT vs BERT Intuition
Why this matters

Modern LLMs like GPT and BERT power chatbots, search, and generative tools. To use them well, you need to grasp how they differ: one predicts the next word (autoregressive), the other predicts masked words (bidirectional).

Theory Essentials

Transformer Recap: self-attention + feed-forward + residual connections.

GPT (Decoder-only): autoregressive; predicts the next token given past context.

BERT (Encoder-only): bidirectional; learns from all context by masking random tokens.

Training Objectives:

GPT → Causal LM (maximize likelihood of next token).

BERT → Masked LM (recover masked words).

Use cases: GPT → text generation; BERT → classification, Q&A, embeddings.

In [ ]:
# Setup
import torch, transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer, BertTokenizer, BertForMaskedLM

# GPT-2 example (autoregressive)
gpt_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2")

text = "The future of AI is"
inputs = gpt_tok(text, return_tensors="pt")
outputs = gpt_model.generate(**inputs, max_new_tokens=10, pad_token_id=gpt_tok.eos_token_id)
print("GPT-2 completion:\n", gpt_tok.decode(outputs[0]))

# BERT example (masked LM)
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased")

masked = "The future of [MASK] is bright."
inputs = bert_tok(masked, return_tensors="pt")
with torch.no_grad():
    logits = bert_model(**inputs).logits
mask_token_index = (inputs.input_ids == bert_tok.mask_token_id)[0].nonzero(as_tuple=True)[0]
pred_id = logits[0, mask_token_index].argmax(axis=-1)
print("BERT fill-mask:\n", masked.replace("[MASK]", bert_tok.decode(pred_id)))


GPT-2 completion:
 The future of AI is uncertain. The future of AI is uncertain.



Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERT fill-mask:
 The future of mankind is bright.


Awesome—Week 13 · Day 1 is exactly this: **GPT (decoder-only, autoregressive) vs BERT (encoder-only, bidirectional).**
Here’s the quick, practical mental model + fixes for your warnings.

# GPT vs BERT — what actually differs

| Aspect                | **GPT (e.g., GPT-2)**                                                            | **BERT (e.g., bert-base-uncased)**                                                                                               |
| --------------------- | -------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| Architecture          | **Decoder-only** Transformer blocks                                              | **Encoder-only** Transformer blocks                                                                                              |
| Attention direction   | **Causal (left-to-right)** — each token sees only past tokens (uses causal mask) | **Bidirectional** — each token attends to **both left & right** context (no causal mask, but uses padding masks)                 |
| Pretraining objective | **Causal LM**: predict next token given previous tokens                          | **Masked LM (MLM)**: randomly mask \~15% tokens and predict them (+ often Next Sentence Prediction in original BERT pretraining) |
| Strengths             | **Generate** coherent continuations; great for writing/code completion           | **Understand**/encode text; great for classification, NER, QA (with head), retrieval encoders                                    |
| Typical fine-tunes    | Instruction tuning, chat, code gen, summarization                                | Add a task head: classification, token classification, QA (span start/end)                                                       |
| Token flow            | One direction → good for generation continuity                                   | Full context per token → strong representations/understanding                                                                    |

**Why GPT repeats sometimes (like your “uncertain…uncertain”)**
Short `max_new_tokens`, greedy decoding (no sampling), and small base model cause loops. Add sampling (`do_sample=True`, `top_p`/`temperature`) and penalties (`repetition_penalty`, `no_repeat_ngram_size`).

**What you should see:**

* GPT-2: a more varied continuation (less loopiness).
* BERT: a sensible single-token replacement (e.g., *“mankind/humanity/progress”*).

---

# When to pick which (rule of thumb)

* **Need to *generate*** text/code/dialogue → **GPT-style**.
* **Need to *understand/classify/search*** text → **BERT-style** (or distilled variants like `distilbert`).
* For many production apps today: **encoder (BERT) for retrieval/ranking** + **decoder (GPT) for generation** (this is also the heart of many RAG systems).


1) Core (10–15 min)
Task: Run the code and explain the difference in outputs.

GPT → continuation, BERT → fill-in

GPT: "The future of AI is uncertain. The future of AI is uncertain."

 BERT: "The future of mankind is bright"


🔹 Why repetition happens

Small model (GPT-2): doesn’t have strong long-range control.

Greedy decoding: it picks the same safe token sequence again and again, leading to loops.

Short prompt: with “The future of AI is”, the model falls into a high-probability phrase like “uncertain” and keeps repeating it.

No penalties: nothing in your call tells GPT to avoid repeats.

This is very common in raw GPT-2.

2) Practice (10–15 min)
Task: Try 3 different prompts with GPT and 3 masked sentences with BERT. Record patterns.

In [ ]:
for text in ["The stock market is about to", "Pyhton is recommendable for", "The next MMA lightweight champion will be" ]:

    inputs = gpt_tok(text, return_tensors="pt")
    outputs = gpt_model.generate(**inputs, max_new_tokens=10, pad_token_id=gpt_tok.eos_token_id)
    print("GPT-2 completion:\n", gpt_tok.decode(outputs[0]))



for masked in ["Humanity is currently [MASK] and it's not good.", "Aristotle told me [MASK] is the best virtue.", "I would recommend any student to study [MASK] as it is the future."]:

    inputs = bert_tok(masked, return_tensors="pt")
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    mask_token_index = (inputs.input_ids == bert_tok.mask_token_id)[0].nonzero(as_tuple=True)[0]
    pred_id = logits[0, mask_token_index].argmax(axis=-1)
    print("BERT fill-mask:\n", masked.replace("[MASK]", bert_tok.decode(pred_id)))


GPT-2 completion:
 The stock market is about to go up, and the stock market is about to
GPT-2 completion:
 Pyhton is recommendable for a lot of reasons.

The first is
GPT-2 completion:
 The next MMA lightweight champion will be announced on July 1, 2017.

The
BERT fill-mask:
 Humanity is currently dying and it's not good
BERT fill-mask:
 Aristotle told me virtue is the best virtue.
BERT fill-mask:
 I would recommend any student to study history as it is the future.


3) Stretch (optional, 10–15 min)
Task: Modify GPT’s generation with temperature=1.2 and top_k=30. Compare outputs.

In [9]:
gpt_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2")

for text in ["The future will be ", "Learning AI is very", "Humanity will colonize the stars when" ]:

    inputs = gpt_tok(text, return_tensors="pt")
    outputs = gpt_model.generate(**inputs, max_new_tokens=10,
                             temperature=1.2, top_k=30, pad_token_id=gpt_tok.eos_token_id)

    print("GPT-2 completion:\n", gpt_tok.decode(outputs[0]))

GPT-2 completion:
 The future will be  a world where the world is not a place
GPT-2 completion:
 Learning AI is very important to us. We need to understand how to
GPT-2 completion:
 Humanity will colonize the stars when they are in the right place.

The


Mini-Challenge (≤40 min)

Build a small notebook demo comparing GPT vs BERT.

Input: user text.

If no [MASK] → use GPT to continue.

If [MASK] present → use BERT to fill.

Acceptance Criteria

Handles at least 3 user examples.

Prints clear outputs side by side.

Short note: when to use GPT vs BERT.

In [10]:
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
gpt_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2")



def demo(text: str) -> str:
    """If text has [MASK] → fill with BERT. Else → continue with GPT-2."""
    text = text.strip()
    if "[MASK]" in text:
        # BERT fill (handles first [MASK])
        enc = bert_tok(text, return_tensors="pt")
        with torch.no_grad():
            logits = bert_model(**enc).logits
        mask_idx = (enc.input_ids[0] == bert_tok.mask_token_id).nonzero(as_tuple=True)[0].item()
        pred_id = logits[0, mask_idx].argmax().item()
        return text.replace("[MASK]", bert_tok.decode([pred_id]))
    else:
        # GPT-2 continuation (simple, non-greedy to avoid loops)
        enc = gpt_tok(text, return_tensors="pt")
        out = gpt_model.generate(
            **enc,
            max_new_tokens=40,
            do_sample=True, top_p=0.95, temperature=0.9,
            repetition_penalty=1.1, no_repeat_ngram_size=3,
            pad_token_id=gpt_tok.eos_token_id
        )
        return gpt_tok.decode(out[0], skip_special_tokens=True)

# ---- Simple interactive loop (3 examples) ----
for i in range(1, 4):
    user_in = input(f"Example {i}/3 — Enter text (use [MASK] to trigger BERT): ").strip()
    out = demo(user_in)
    print("\n--- INPUT ---")
    print(user_in)
    print("\n--- OUTPUT ---")
    print(out)
    print("\n" + "="*60 + "\n")

print("Tip: Use '[MASK]' for BERT fill; no mask → GPT-2 continuation.")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



--- INPUT ---
I love [MASK] it is the best sport ever.

--- OUTPUT ---
I love that it is the best sport ever.



--- INPUT ---
Humans will colonize jupyter's moons in the year

--- OUTPUT ---
Humans will colonize jupyter's moons in the year 2000, and many of them may survive to form sentient organisms with which they could exchange information. However—if their food supply were exhausted by that time or if there was no more fuel left for humans



--- INPUT ---
The end of humanity will come when

--- OUTPUT ---
The end of humanity will come when we all start speaking, and it's not long before they're taking us back to Earth."
- "We'll always remember you as the only one who survived on Mars...we were doomed by


Tip: Use '[MASK]' for BERT fill; no mask → GPT-2 continuation.


Notes / Key Takeaways

GPT = decoder-only, left-to-right.

BERT = encoder-only, bidirectional.

GPT good for generation; BERT strong at understanding.

Different pretraining objectives drive capabilities.

Hugging Face makes testing easy with just a few lines.

Reflection

In your own words, why does GPT need a causal mask?

What tasks benefit from bidirectional context instead of left-to-right?

-GPT needs a casual mask so it is can only predict the next token based on previous ones and not 'looking into the future'.

-Used more to understand/classify/ serach text. 